# Compare policy performance

Reads a simulation log (one row per epoch x day x product, written by the policy notebooks)
and summarises each epoch and product.

In [1]:
import polars as pl
from pathlib import Path

# Relative to this notebook (code/sim/) -> results/ at the repo root.
RESULTS = Path('../../results')
LOG_FILE = RESULTS / 'Newsvendor_SingleSupplier_DP_Daily.parquet'

sim_log = pl.read_parquet(LOG_FILE)
sim_log.head()

epoch,seed,day,product,demand,sold,price,wastage,order_qty,order_cost,stock
i32,i64,i32,str,i64,i64,f64,i64,i64,f64,i64
0,0,0,"""FOODS_1""",1305,1259,2.18,0,null,null,69362
0,0,0,"""FOODS_2""",1073,1024,2.8,0,null,null,95552
0,0,0,"""FOODS_3""",7602,6294,2.16,0,61000,22570.0,53075
0,0,0,"""HOBBIES_1""",1798,1461,2.87,0,null,null,668159
0,0,0,"""HOBBIES_2""",36,31,1.73,0,null,null,34518


## Performance per epoch and product

* **revenue** = units sold x the price they sold at
* **stockout_days** = days that end with no stock on hand
* **lost_units** on a stockout day = expected buyers - units sold. Expected buyers = demand x
  the share of customers who bought at that price on days the product was in stock (same epoch,
  product and price; falls back to the epoch-product share if that price was never in stock).
* **underage_cost** = lost_units x unit margin, where unit margin = price - average unit cost of
  that epoch's orders (floored at 0).

In [2]:
def performance(log):
    """Totals per (epoch, product) from a daily simulation log."""
    log = log.with_columns(
        (pl.col('stock') == 0).alias('stockout'),
        (pl.col('sold') * pl.col('price')).alias('revenue'),
    )

    # Share of demand that buys at a price, measured on in-stock days (demand there is not
    # capped by stock). Per epoch/product/price, with an epoch/product fallback.
    in_stock = log.filter(~pl.col('stockout') & (pl.col('demand') > 0))
    share_at_price = in_stock.group_by('epoch', 'product', 'price').agg(
        (pl.col('sold').sum() / pl.col('demand').sum()).alias('share_price')
    )
    share_overall = in_stock.group_by('epoch', 'product').agg(
        (pl.col('sold').sum() / pl.col('demand').sum()).alias('share_overall')
    )

    # Average unit cost of everything ordered in the epoch (null if nothing was ordered).
    unit_cost = log.group_by('epoch', 'product').agg(
        (pl.col('order_cost').sum() / pl.col('order_qty').sum()).alias('unit_cost')
    )

    shortfall = (
        log.filter(pl.col('stockout'))
        .join(share_at_price, on=['epoch', 'product', 'price'], how='left')
        .join(share_overall, on=['epoch', 'product'], how='left')
        .join(unit_cost, on=['epoch', 'product'], how='left')
        .with_columns(
            (pl.col('demand') * pl.coalesce('share_price', 'share_overall') - pl.col('sold'))
            .clip(lower_bound=0).alias('lost_units'),
            (pl.col('price') - pl.col('unit_cost').fill_null(0)).clip(lower_bound=0).alias('unit_margin'),
        )
        .group_by('epoch', 'product')
        .agg(
            pl.col('lost_units').sum(),
            (pl.col('lost_units') * pl.col('unit_margin')).sum().alias('underage_cost'),
        )
    )

    return (
        log.group_by('epoch', 'product')
        .agg(
            pl.col('demand').sum().alias('total_demand'),
            pl.col('sold').sum().alias('total_sold'),
            pl.col('revenue').sum().alias('total_revenue'),
            pl.col('wastage').sum().alias('total_wastage'),
            pl.col('order_qty').sum().alias('total_order_qty'),      # nulls (no order) skipped
            pl.col('order_cost').sum().alias('total_cost'),
            pl.col('stockout').sum().alias('stockout_days'),
        )
        .join(shortfall, on=['epoch', 'product'], how='left')
        .with_columns(pl.col('lost_units', 'underage_cost').fill_null(0))  # never out of stock
        .sort('epoch', 'product')
    )


perf = performance(sim_log)
perf

epoch,product,total_demand,total_sold,total_revenue,total_wastage,total_order_qty,total_cost,stockout_days,lost_units,underage_cost
i32,str,i64,i64,f64,i64,i64,f64,u32,f64,f64
0,"""FOODS_1""",2541283,1528755,3.7724e6,25878,1528000,491565.0,7,4092.947308,8792.858879
0,"""FOODS_2""",2002499,1193934,3.8980e6,36673,1156000,554030.0,18,11556.247433,31499.725407
0,"""FOODS_3""",12475477,7758963,1.7061e7,2459593,10230750,3.7300e6,81,269877.012577,502030.831699
0,"""HOBBIES_1""",2763083,1259832,4.5708e6,53758,695000,305800.0,6,916.205005,3811.412822
0,"""HOBBIES_2""",227900,101849,224761.3,1109,90000,30500.0,0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…
99,"""FOODS_3""",12543597,7429443,1.6499e7,1992972,9411080,3.5128e6,87,276278.104026,520980.388585
99,"""HOBBIES_1""",2768660,1204642,4.4289e6,48921,682500,329300.0,11,745.459763,3285.620733
99,"""HOBBIES_2""",226320,102486,225597.12,0,90000,22500.0,0,0.0,0.0


## Mean and standard deviation over epochs, per product

Each metric becomes two columns, `<metric>_mean` and `<metric>_std`, side by side. The std is
the spread across epochs (sample std, ddof=1).

In [3]:
metrics = [c for c in perf.columns if c not in ('epoch', 'product')]


def mean_std(df, by):
    """Mean and std across epochs of every metric, grouped by `by`."""
    return (
        df.group_by(by)
        .agg(
            *[pl.col(m).mean().alias(f'{m}_mean') for m in metrics],
            *[pl.col(m).std().alias(f'{m}_std') for m in metrics],
        )
        .select(*by, *[f'{m}_{s}' for m in metrics for s in ('mean', 'std')])   # mean next to std
        .sort(by[::-1])
    )


summary = mean_std(perf, ['product'])
summary

product,total_demand_mean,total_demand_std,total_sold_mean,total_sold_std,total_revenue_mean,total_revenue_std,total_wastage_mean,total_wastage_std,total_order_qty_mean,total_order_qty_std,total_cost_mean,total_cost_std,stockout_days_mean,stockout_days_std,lost_units_mean,lost_units_std,underage_cost_mean,underage_cost_std
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""FOODS_1""",2.5503e6,5370.751681,1.5424e6,20594.912532,3.7964e6,38126.672188,21376.91,4219.396889,1.559475e6,28451.924275,491137.1,11502.740191,13.22,9.329631,8025.458103,6242.644817,17283.787906,13411.543919
"""FOODS_2""",2004525.2,6189.524446,1.1990e6,6349.690478,3.9133e6,20515.191257,39005.57,6946.679874,1.16382e6,17824.837965,588721.9,20715.808324,12.69,7.763805,6156.254886,4781.760184,17163.463352,13211.246924
"""FOODS_3""",1.2497e7,27288.330069,7.3556e6,103318.428073,1.6358e7,190330.940009,1.9995e6,189503.091721,9349317.2,238955.342935,3.4806e6,81942.670806,79.61,15.703628,271479.321402,59738.015642,512240.809064,112800.602774
"""HOBBIES_1""",2.7682e6,5197.752827,1.2614e6,30811.499516,4.5710e6,74391.756802,58725.5,7017.472877,682890.0,26749.92495,319123.25,19080.010805,8.93,4.295299,3438.972688,3271.82322,12178.267424,10633.826826
"""HOBBIES_2""",226987.01,941.009103,103861.97,2817.154157,227228.9795,4705.722999,3566.24,3426.07471,91661.0,10907.414724,25908.71,4245.696638,3.98,9.225546,365.78242,898.718469,707.770814,1732.678566
"""HOUSEHOLD_1""",4347166.3,14883.767804,2.5020e6,30588.647725,9.4935e6,108420.084711,0.0,0.0,2.00858e6,71334.633237,1117056.5,39633.799405,5.59,5.792881,8023.03817,8754.356108,26028.987009,28405.879391
"""HOUSEHOLD_2""",1.5863e6,3372.532353,934011.47,13895.909561,4.7642e6,68535.706845,0.0,0.0,705700.0,27895.005165,514105.2,20392.256328,3.9,6.679533,2040.310435,3525.822328,8944.231327,15455.516008


## Save to Excel

Both tables, rounded to 2 decimals, in one workbook in `results/`: sheet `per_epoch` (per epoch
and product) and sheet `summary` (mean and std over epochs).

In [4]:
import polars.selectors as cs
from xlsxwriter import Workbook

excel_file = RESULTS / f'performance_{LOG_FILE.stem}.xlsx'
with Workbook(excel_file) as wb:
    perf.with_columns(cs.float().round(2)).write_excel(wb, worksheet='per_epoch')
    summary.with_columns(cs.float().round(2)).write_excel(wb, worksheet='summary')
print(f'saved {excel_file.resolve()}')

saved C:\Users\prade\OneDrive\Documents\Code\retail-revenue-management\results\performance_Newsvendor_SingleSupplier_DP_Daily.xlsx


## Compare every policy in results/

The policy name is taken from the file name.

In [ ]:
all_perf = pl.concat([
    performance(pl.read_parquet(f)).with_columns(pl.lit(f.stem).alias('policy'))
    for f in sorted(RESULTS.glob('*.parquet'))
])

comparison = mean_std(all_perf, ['policy', 'product'])   # sorted by product, then policy
comparison